In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
api_token="oAB7Iu9rkv183ffixxkRTm7uAl1HL5OH_EUbDAyBt8dRRFDDhc3sTtuYwL7TgYiK"
username="eckkam11"

In [ ]:
%run /content/drive/MyDrive/treedata/Scripts/USGS_pipe.py

USGS username: eckkam11
M2M token: ··········
✓ Logged in

Dataset              Year   Scenes   Combined Cov%   Bands  Band Type              Resolution
----------------------------------------------------------------------------------------------------
NAIP                 2013   4        100.0%          4      CNIR                   1.00 ✓
NAIP                 2015   4        100.0%          4      CNIR                   1.00 ✓
NAIP                 2017   4        100.0%          4      CNIR                   1.00 ✓
NAIP                 2019   4        100.0%          4      CNIR                   0.60 ✓
NAIP                 2023   4        100.0%          4      CNIR                   0.60 ✓
HIGH_RES_ORTHO       2012   79       100.0%          3      Color                  0.304803 ✓

Years with ≥80% combined coverage:
  HIGH_RES_ORTHO       2012
  NAIP                 2013, 2015, 2017, 2019, 2023

✓ Saved → /content/edmonds_session/modern_inventory.csv
✓ Logged out


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import numpy as np
from pathlib import Path

MOSAIC_DIR   = Path('/content/edmonds_session/mosaics')
GEOREF_DIR   = Path('/content/edmonds_session/georeferenced')
BOUNDARY     = Path('/content/drive/MyDrive/treedata/City Boundry/Edmonds Boundry.shp')

boundary_gdf = gpd.read_file(BOUNDARY)
if boundary_gdf.crs and str(boundary_gdf.crs) != 'EPSG:4326':
    boundary_wgs84 = boundary_gdf.to_crs('EPSG:4326')
else:
    boundary_wgs84 = boundary_gdf

# Build year → image path mapping
# Prefer georeferenced version, fall back to mosaic
year_paths = {}
for p in sorted(MOSAIC_DIR.iterdir()):
    if p.is_dir():
        m = p / f'{p.name}_mosaic.tif'
        if m.exists():
            year_paths[int(p.name)] = ('mosaic', m)

for p in sorted(GEOREF_DIR.glob('*_georef.tif')):
    # Extract year from tracking
    import re
    # Get year from the mosaic dir structure by matching display_id
    # We'll open and check the CRS instead
    year_paths_georef = {}

# Simpler: just show all georeferenced files + all mosaics separately
georef_files = sorted(GEOREF_DIR.glob('*_georef.tif'))
mosaic_years = sorted([int(p.name) for p in MOSAIC_DIR.iterdir() if p.is_dir()])

# Build combined list: georef files first, then remaining mosaic years
all_items = []
for gf in georef_files:
    all_items.append(('georef', gf.stem.replace('_georef',''), gf))
for yr in mosaic_years:
    mp = MOSAIC_DIR / str(yr) / f'{yr}_mosaic.tif'
    if mp.exists():
        all_items.append(('mosaic', str(yr), mp))

cols = 4
rows = (len(all_items) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for ax, (kind, label, fpath) in zip(axes, all_items):
    ax.set_facecolor('#0d1117')
    ax.axis('off')
    tag = '✓ georef' if kind == 'georef' else 'mosaic'
    ax.set_title(f'{label}\n{tag}', color='white' if kind == 'mosaic' else '#00e676',
                 fontsize=9, pad=3)
    try:
        with rasterio.open(fpath) as src:
            has_geo = src.crs is not None
            scale   = 0.05
            h = max(1, int(src.height * scale))
            w = max(1, int(src.width  * scale))
            bands = min(src.count, 3)
            data = src.read(list(range(1, bands + 1)), out_shape=(bands, h, w))
            img = np.transpose(data, (1, 2, 0)).astype(float)
            p2, p98 = np.percentile(img, 2), np.percentile(img, 98)
            img = np.clip((img - p2) / max(p98 - p2, 1), 0, 1)
            if bands == 1:
                img = np.stack([img[:,:,0]] * 3, axis=-1)

            if has_geo:
                extent = [src.bounds.left, src.bounds.right,
                          src.bounds.bottom, src.bounds.top]
                ax.imshow(img, extent=extent, origin='upper', aspect='equal')
                try:
                    bnd = boundary_wgs84.to_crs(src.crs)
                    bnd.boundary.plot(ax=ax, color='red', linewidth=1.5)
                except Exception:
                    boundary_wgs84.boundary.plot(ax=ax, color='red', linewidth=1.5)
            else:
                ax.imshow(img, origin='upper', aspect='equal')
                ax.text(0.5, 0.02, 'no georeference', transform=ax.transAxes,
                        color='#ff6b6b', fontsize=7, ha='center')
    except Exception as e:
        ax.text(0.5, 0.5, str(e)[:80], transform=ax.transAxes,
                color='#ff6b6b', fontsize=7, ha='center', va='center', wrap=True)

for ax in axes[len(all_items):]:
    ax.set_visible(False)

plt.tight_layout(pad=0.5)
out = '/content/edmonds_session/georef_overview.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Saved → {out}')